# World Cup Sentiment Tracker

## Data Exploration **(Real Tweets)** 

---

### First Dataset:

Source type:  
Replies/comments under official football tweets  
(less likely to be biased and/or noisy since official posts are usually neutral/promotional, but the replies are fan reactions)

Accounts:
- @FIFAWorldCup
- national team accounts
- major player/team accounts if relevant

Tweet types:
- goal announcements
- full-time result posts
- VAR/penalty/red-card posts
- lineup posts
- highlight clips

Collected unit:  
Individual reply/comment, not the official post itself

----
### Strategy:

1. Pick one match/event
2. Find official posts about that event
3. Pull replies under those posts
4. Store replies with metadata
5. Run baseline sentiment
6. Manually inspect model failures

---

## Real Football Tweet Collection

Goal:
Collect replies from official football event posts and evaluate how well the baseline sentiment model performs on real football fan language.

Questions:
1. Can we successfully collect real football discussion?
2. Does the baseline sentiment model perform reasonably well?
3. What types of football language cause model failures?

In [1]:
import sys
sys.path.append("../src")

import pandas as pd

from api.getxapi_client import (
    fetch_replies,
    normalize_replies
)
from data.preprocess import (
    annotate_replies,
    preprocess_replies,
    preprocessing_summary
)
from models.sentiment_baseline import (
    add_sentiment,
    load_sentiment_model
)

In [2]:
response = fetch_replies(tweet_id="2069468618476200189")

response.keys()

dict_keys(['tweetId', 'reply_count', 'has_more', 'next_cursor', 'replies'])

In [3]:
rows = normalize_replies(
    response=response,
    match="Portugal vs Spain",
    event="goal",
    source_account="FIFAWorldCup",
    team="Portugal"
)

raw_df = pd.DataFrame(rows)

raw_df.head()

,tweet_id,parent_tweet_id,url,timestamp,text,author_username,author_name,like_count,reply_count,view_count,match,team,player,event,source_account,source
0,2069547886975635584,2069468618476200189,https://x.com/NoCaptionMood/status/20695478869...,Tue Jun 23 22:27:21 +0000 2026,@FIFAWorldCup https://x.com/i/status/206952949...,NoCaptionMood,No Caption Mood,71,1,20238,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
1,2069547912825049173,2069468618476200189,https://x.com/grok/status/2069547912825049173,Tue Jun 23 22:27:27 +0000 2026,@NoCaptionMood @FIFAWorldCup Ask Grok is curre...,grok,Grok,111,1,11446,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
2,2069474215783215339,2069468618476200189,https://x.com/FOXSoccer/status/206947421578321...,Tue Jun 23 17:34:36 +0000 2026,"After VAR review, the Uzbekistan goal is disal...",FOXSoccer,FOX Soccer,552,29,1217553,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
3,2069469280991637775,2069468618476200189,https://x.com/DPicwin/status/2069469280991637775,Tue Jun 23 17:15:00 +0000 2026,"@FIFAWorldCup No one—I repeat, no one can ever...",DPicwin,D. Picwin,445,17,18166,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
4,2069468946906886206,2069468618476200189,https://x.com/Shameless_925/status/20694689469...,Tue Jun 23 17:13:40 +0000 2026,@FIFAWorldCup Stupid record. Messi has 18 WC g...,Shameless_925,Amanda❤️🌸,45,38,19378,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi


In [4]:
raw_df.columns

Index(['tweet_id', 'parent_tweet_id', 'url', 'timestamp', 'text',
       'author_username', 'author_name', 'like_count', 'reply_count',
       'view_count', 'match', 'team', 'player', 'event', 'source_account',
       'source'],
      dtype='str')

In [5]:
raw_df.to_csv(
    "../data/raw/real_replies_sample.csv",
    index=False
)

In [6]:
raw_df.shape

(38, 16)

In [7]:
raw_df["text"].head(20)

0     @FIFAWorldCup https://x.com/i/status/206952949...
1     @NoCaptionMood @FIFAWorldCup Ask Grok is curre...
2     After VAR review, the Uzbekistan goal is disal...
3     @FIFAWorldCup No one—I repeat, no one can ever...
4     @FIFAWorldCup Stupid record. Messi has 18 WC g...
5     @FIFAWorldCup Messi's fans right now🤣🤣🤣 https:...
6     @FIFAWorldCup CRISTIANO RONALDO IS THE NAME!!\...
7     @FIFAWorldCup When Ronaldo scores\nThe whole w...
8     @FIFAWorldCup Relax… it’s just against Uzbekis...
9     @FIFAWorldCup @windrawwin Absolute scenes!   F...
10    @FIFAWorldCup He has won how many World cup tr...
11    First 2026 FIFA World Cup goal for Rafael Leão...
12    @FIFAWorldCup Another record set! https://t.co...
13    @FIFAWorldCup 📹 credit:FIFA https://t.co/wHEFC...
14    @FIFAWorldCup Another record https://t.co/u32B...
15    @FIFAWorldCup @BigDee_UTD https://t.co/oXdQR8BwAk
16    @FIFAWorldCup the GREATEST🫶🏻♾️ https://t.co/Q8...
17    @FIFAWorldCup Six World Cups and still mak

In [8]:
annotated_df = annotate_replies(raw_df)
analysis_df = preprocess_replies(raw_df)

preprocessing_summary(annotated_df, analysis_df)

{'raw_rows': 38,
 'analysis_rows': 11,
 'removed_rows': 27,
 'filter_reasons': {'low_relevance': 21, 'keep': 11, 'unusable_text': 6}}

In [10]:
annotated_df.to_csv(
    "../data/processed/annotated_replies_sample.csv",
    index=False)

In [9]:
annotated_df["filter_reason"].value_counts()

filter_reason
low_relevance    21
keep             11
unusable_text     6
Name: count, dtype: int64

In [11]:
analysis_df[
    [
        "text",
        "clean_text",
        "relevance_score",
        "filter_reason"
    ]
].head(20)

,text,clean_text,relevance_score,filter_reason
0,"After VAR review, the Uzbekistan goal is disal...","After VAR review, the Uzbekistan goal is disal...",5,keep
1,@FIFAWorldCup @windrawwin Absolute scenes! F...,Absolute scenes! First man to score in 6 World...,1,keep
2,@FIFAWorldCup He has won how many World cup tr...,He has won how many World cup trophies? 😂😂,2,keep
3,First 2026 FIFA World Cup goal for Rafael Leão...,First 2026 FIFA World Cup goal for Rafael Leão 👏,5,keep
4,@FIFAWorldCup People spent years saying it cou...,People spent years saying it could never be do...,5,keep
5,@FIFAWorldCup The only player to score in six ...,The only player to score in six world cups wit...,2,keep
6,@FIFAWorldCup How many total World Cup goals d...,How many total World Cup goals does he have? S...,2,keep
7,The world's game just upgraded its operating s...,The world's game just upgraded its operating s...,3,keep
8,@FIFAWorldCup Six World Cups. Six tournaments ...,Six World Cups. Six tournaments with a goal. C...,4,keep
9,@FIFAWorldCup Ronaldo score his first goal of ...,Ronaldo score his first goal of the World Cup....,8,keep


In [ ]:
classifier = load_sentiment_model()

analysis_df = add_sentiment(
    analysis_df,
    classifier=classifier
)

analysis_df.head()

In [ ]:
sample = analysis_df.sample(
    min(25, len(analysis_df)),
    random_state=42
)

sample[
    [
        "text",
        "clean_text",
        "sentiment_label",
        "sentiment_score",
        "relevance_score"
    ]
]

### Observations

Real reply data contains media-only, text-poor, spam-like, and low-relevance replies.

The notebook now treats this as a reusable preprocessing step rather than notebook-only cleanup:

```text
Raw API replies
-> normalized rows
-> data quality filtering
-> baseline sentiment
```

Filtering is implemented in `src/data/preprocess.py`, so future notebooks and dashboards can reuse the same cleaning, audit, and relevance-scoring logic.